Notebook creates & saves basic hypothesis sets for PRC, plus performs wald test. 

In [42]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd

In [43]:
from pathlib import Path

In [44]:
DATA_ROOT=Path("/gpfs/gibbs/pi/reilly/tabula_data")
path=DATA_ROOT/"simulated"
name="shendure_calibrated_sim_with_orthos_20251116"

In [45]:
demo_counts=scm.scMPRA_data.from_parquet(path/name/"scMPRA/0.scmpra")
demo_counts.ortho_filter()

scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [46]:
# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=demo_counts,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=demo_counts,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)

In [47]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=8,#cores per slurm job
        memory="80G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=1:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=3)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

In [48]:
from dask.distributed import Semaphore, as_completed, get_client

In [49]:
ortho_root=path/name/"orthos_with_precomputed_wald_erin_numerical_stability_test"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

Semaphore(max_leases=3, name="test")

def compute_one_wald(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    sem = Semaphore(name="test")
    with sem:
        client=get_client()
        ortho_oi=scm.ortho.load(client=client,
                                    path=input_root,
                                    name=name)
        runner = scm.HypothesisTester(test_type)
        output_short=Path(output_root)/hypothesis_set_name/test_type
        output_short.mkdir(exist_ok=True,parents=True)
        runner.run(hypothesis_set, ortho_oi, client).to_tsv(output_short/name)



In [50]:
input_ortho_names

['0', '1', '2', '3', '4']

In [52]:
client=get_client()
ortho_oi_0=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=input_ortho_names[0])

ortho_oi_0.by_cell_type.model['ExEndodermVisceral'].result()

{'llf_total': -156218.59066743497,
 'llfs': array([-156218.59066743]),
 'aic_total': 312747.18133486994,
 'aics': array([312747.18133487]),
 'df_model_total': 155,
 'df': 155,
 'weights': {'x_mu': Intercept                                                           -4.072576
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8174]      2.406501
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8175]      3.558702
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8179]      1.916069
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8192]     -0.523850
                                                                         ...   
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7975]    0.263586
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7978]    3.331725
  C(cre_id, contr.treatment(base='reference'))[T.eef1aP]               9.211071
  C(cre_id, contr.treatment(base='reference'))[T.pgk1P]                7.860569
  C(c

In [53]:
ortho_oi_3=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=input_ortho_names[3])

ortho_oi_3.by_cell_type.model['ExEndodermVisceral'].result()

{'llf_total': -164705.71475047292,
 'llfs': array([-164705.71475047]),
 'aic_total': 329721.42950094584,
 'aics': array([329721.42950095]),
 'df_model_total': 155,
 'df': 155,
 'weights': {'x_mu': Intercept                                                            132.952835
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8174]     -134.296768
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8175]     -133.131744
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8179]     -134.530838
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8192]     -137.841385
                                                                          ...    
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7975]   -136.514267
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7978]   -133.382172
  C(cre_id, contr.treatment(base='reference'))[T.eef1aP]              -127.726570
  C(cre_id, contr.treatment(base='reference'))[T.pgk1P]            

In [11]:
for rep in (input_ortho_names):
    print("Simulation replicate: %s" % rep)
    ortho_oi=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=rep)
    for ct in ortho_oi.by_cell_type.model.keys():
        print("%s llf_total: %s" % (ct, ortho_oi.by_cell_type.model[ct].result()['llf_total']))

    print()

Simulation replicate: 0
Cardiomyocytes llf_total: -29359.958255389123
EpiblastPrimitiveStreak llf_total: -222080.85165495798
ExEndodermParietal llf_total: -322062.0053414176
ExEndodermVisceral llf_total: -156218.59066743497
Haematoendothelial llf_total: -47923.173563371645
Mesoderm llf_total: -287664.7529560905
NeuroectodermBrain llf_total: -303373.1402040655
NeuroectodermRostral llf_total: -60294.77267020178
SurfaceEctoderm llf_total: -168039.09528065985
reference llf_total: -605025.7000021078

Simulation replicate: 1
Cardiomyocytes llf_total: -28647.227658434538
EpiblastPrimitiveStreak llf_total: -219811.89582360908
ExEndodermParietal llf_total: -321301.3699642923
ExEndodermVisceral llf_total: -157663.31323151663
Haematoendothelial llf_total: -48387.85003270302
Mesoderm llf_total: -290033.90228405315
NeuroectodermBrain llf_total: -299767.1379107144
NeuroectodermRostral llf_total: -59953.78011478431
SurfaceEctoderm llf_total: -169998.3115181243
reference llf_total: -613727.033849217



In [16]:
#hs_all_ct
futures_ct = [client.submit(compute_one_wald,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_ct,
                        hypothesis_set_name="hs_all_ct",
                        test_type="wald") for name_oi in input_ortho_names]

In [17]:
#hs_all_cre
futures_cre = [client.submit(compute_one_wald,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_cre,
                        hypothesis_set_name="hs_all_cre",
                        test_type="wald") for name_oi in input_ortho_names]

In [18]:
scmpradat_root=path/name/"scMPRA"
def compute_one_mwu(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    sem = Semaphore(name="test")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        runner = scm.HypothesisTester(test_type)
        output_short=Path(output_root)/hypothesis_set_name/test_type
        output_short.mkdir(exist_ok=True,parents=True)
        runner.run(hypothesis_set, dat, client).to_tsv(output_short/name)

In [19]:
futures_mwu_ct = [client.submit(compute_one_mwu,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_ct,
                        hypothesis_set_name="hs_all_ct",
                        test_type="mwu") for name_oi in input_ortho_names]

In [25]:
futures_mwu

[<Future: finished, type: NoneType, key: compute_one_mwu-481690e02f12404546339b53e3ee0990>,
 <Future: finished, type: NoneType, key: compute_one_mwu-e83e853670f27bf7ec45b2a7f6d0c205>,
 <Future: finished, type: NoneType, key: compute_one_mwu-cc048587bc60fb58fc4c86c25b42293d>,
 <Future: pending, key: compute_one_mwu-887a6689f81f8b8d86eb2a507ec1050e>,
 <Future: pending, key: compute_one_mwu-496a06fdeeeac61d451c0da6e7c48173>]

In [21]:
futures

[<Future: finished, type: NoneType, key: compute_one_wald-002bdcbbf8a0c3aad007473a4d22a61c>,
 <Future: finished, type: NoneType, key: compute_one_wald-ea3b2b67f927cb03a4352046b0b81457>,
 <Future: finished, type: NoneType, key: compute_one_wald-55f2a70cc455d365ad5d052123db6984>,
 <Future: finished, type: NoneType, key: compute_one_wald-cbfe0f2c0dada2a79da7fb7bb3621c52>,
 <Future: finished, type: NoneType, key: compute_one_wald-59529f5a766fd60fe984878494043b2a>]

In [36]:
futures_mwu_cre = [client.submit(compute_one_mwu,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_cre,
                        hypothesis_set_name="hs_all_cre",
                        test_type="mwu") for name_oi in input_ortho_names[3:]]

In [31]:
input_ortho_names[3:]

['3', '4']

In [38]:
futures_mwu_cre

[<Future: finished, type: NoneType, key: compute_one_mwu-5d0e7987fc44b5a82fce4ba9bcb75803>,
 <Future: finished, type: NoneType, key: compute_one_mwu-eb3b4b61882d9db57cb7e45c296e0da8>]

In [39]:
ortho_oi.wald_precomp.by_cell_type['Cardiomyocytes'].result().debug_msg

RuntimeError: IOLoop is closed

In [40]:
client.close()
cluster.close()